We have to find what is a good indicator of draft position! So for this, I first need to make a query to find multiple statistics from college receivers:

YPG
YPC
MAX REC TD
40 Time
Overall Pick

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nfl_analytics.db')

query = """
WITH player_data AS (
    SELECT 
        c.Player,
        c.Conf,
        AVG(c.RecYperG) as college_ypg,
        AVG(c.YperR) as yards_per_catch,
        MAX(c.RecTD) as peak_td,
        com.`40yd` as forty_time,
        d.Player AS Pick,
        d.Year as draft_year,
        CASE 
            WHEN c.Conf IN ('SEC', 'Big Ten') THEN 'SEC, BIG 10'
            WHEN c.Conf IN ('ACC', 'Big 12', 'Pac-12') THEN 'ACC, BIG 12, PAC 12'
            ELSE 'Other'
        END as conference_tier
    FROM college c
    LEFT JOIN draft_picks d ON d.Name = c.Player
    LEFT JOIN combine com ON com.Player = c.Player
    WHERE d.Year BETWEEN 2017 AND 2025
        AND com.`40yd` IS NOT NULL  -- Only players with 40 time
        AND com.`40yd` > 0
    GROUP BY c.Player
)
SELECT * FROM player_data
ORDER BY Pick;
"""



# Execute and display
player_data = pd.read_sql_query(query, conn)

print("WR COLLEGE STATS AND PICKS (2017-2025)")
print("=" * 100)
print(player_data.to_string(index=False))


conn.close()

Now we need to plot it on a scatter plot to see what we can find!

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def get_marker_size(pick):
    # BASED ON ROUNDS LAST YEAR
    if pick <= 10:
        return 300  # Top 10: huge
    elif pick <= 32:
        return 250  # Round 1: big
    elif pick <= 64:
        return 200  # Round 2: medium-large
    elif pick <= 102:
        return 150  # Round 3: medium
    elif pick <= 138:
        return 100   # Round 4: small
    elif pick <= 176:
        return 70   # Round 5: small
    elif pick <= 216:
        return 40   # Round 6: small
    else:
        return 20   # Round 7: tiny


print("CREATING SCATTER PLOT: COLLEGE PRODUCTION vs 40 TIME")
print("=" * 60)

# First, let's check what's in player_data
print(f"Total players with data: {len(player_data)}")
print(f"Pick range: {player_data['Pick'].min()} to {player_data['Pick'].max()}")
print("\n" + "=" * 60)

# Set up the plot
fig, ax = plt.subplots(figsize=(14, 10))

# Define colors for conference tiers
conference_colors = {
    'SEC, BIG 10': '#FF4D4D',         # Red
    'ACC, BIG 12, PAC 12': '#4D4DFF', # Blue
    'Other': '#4DFF4D'                # Green
}

# Define marker sizes based on pick number (lower pick = bigger marker)
# Pick 1 = size 300, Pick 300 = size 30
max_pick = player_data['Pick'].max()
player_data['marker_size'] = player_data['Pick'].apply(get_marker_size)

# Create scatter plot for each conference tier
for tier, color in conference_colors.items():
    tier_data = player_data[player_data['conference_tier'] == tier]
    
    if len(tier_data) > 0:
        scatter = ax.scatter(
            tier_data['college_ypg'], 
            tier_data['forty_time'],
            s=tier_data['marker_size'],  # Size by pick number
            c=color,
            alpha=0.6,
            edgecolors='black',
            linewidth=0.5,
            label=f"{tier} ({len(tier_data)} players)"
        )

# Add labels for top 10 picks
top_picks = player_data[player_data['Pick'] <= 10]
for _, player in top_picks.iterrows():
    ax.annotate(
        f"{player['Player']} (#{player['Pick']})",
        (player['college_ypg'], player['forty_time']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=9,
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7)
    )

# Add a colorbar to explain marker sizes
# Create a legend for pick ranges
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

legend_elements = [
    Patch(facecolor='#FF4D4D', alpha=0.7, label='Top 2 Conferences'),
    Patch(facecolor='#4D4DFF', alpha=0.7, label='Power 5'),
    Patch(facecolor='#4DFF4D', alpha=0.7, label='Other'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=15, label='Top 10 Pick'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=10, label='Mid Round'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=5, label='Late Pick')
]

# Customize the chart
ax.set_xlabel('College Yards Per Game', fontsize=12, fontweight='bold')
ax.set_ylabel('40 Yard Dash (seconds)', fontsize=12, fontweight='bold')
ax.set_title('WR Draft: College Production vs Athleticism (Colored by Conference, Size = Pick #)', 
             fontsize=16, fontweight='bold')

# Invert y-axis (lower 40 time is better)
ax.invert_yaxis()

# Add grid
ax.grid(True, alpha=0.3, linestyle='--')

# Add legend
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

# Add some reference lines (optional)
ax.axhline(y=4.5, color='gray', linestyle='--', alpha=0.5, label='4.5 sec threshold')
ax.axvline(x=70, color='gray', linestyle='--', alpha=0.5, label='70 YPG threshold')

plt.tight_layout()
plt.savefig('wr_draft_scatter_by_pick.png', dpi=300, bbox_inches='tight')
plt.show()

# Show summary statistics
print("\n📊 SUMMARY BY CONFERENCE TIER:")
print("=" * 60)
summary = player_data.groupby('conference_tier').agg({
    'Player': 'count',
    'college_ypg': 'mean',
    'forty_time': 'mean',
    'Pick': 'mean'
}).round(2)
summary.columns = ['Count', 'Avg YPG', 'Avg 40 Time', 'Avg Pick']
print(summary.to_string())

print("\n✅ Visualization saved as 'wr_draft_scatter_by_pick.png'")